# Snowflake Notebook: ポートフォリオ分析

## 分析内容
- 顧客セグメント別の運用成績可視化
- ポートフォリオの分散度ヒートマップ
- リバランスが必要な顧客の抽出（目標比率±5%以上の乖離）
- 業種別パフォーマンス分析

## ポイント
- **Snowparkデータフレーム**: Snowflake上でデータ処理
- **多様な可視化**: Matplotlib、Seaborn、Plotlyを使い分け
- **インタラクティブ性**: Plotlyでドリルダウン可能
- **業務指標**: シャープレシオ（リスクに対するリターンの効率）、ハーフィンダール指数（市場の集中度）など実務的指標
- **統合分析**: 1つのNotebookで包括的な分析を実行

In [ ]:
import snowflake.snowpark as snowpark
from snowflake.snowpark.functions import col, sum, avg, count, round as sf_round, abs as sf_abs
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

print("ライブラリのインポート完了")

## 1. データ確認
SQLセルでスキーマ設定とテーブル確認を行います。

In [ ]:
%%sql -r setup_result
USE ROLE MU;
USE WAREHOUSE MUWH1;
USE SCHEMA SECURITIES_WORKSHOP.ANALYTICS;

In [ ]:
%%sql -r table_info
SELECT table_name, row_count, bytes
FROM SECURITIES_WORKSHOP.INFORMATION_SCHEMA.TABLES 
WHERE table_schema IN ('RAW_DATA', 'ANALYTICS', 'STAGING')
ORDER BY table_schema, table_name;

## 2. 顧客セグメント別 運用成績の可視化
リスク許容度ごとにポートフォリオの運用成績（含み損益率）を比較します。

In [ ]:
%%sql -r portfolio_data
SELECT 
  c.customer_id,
  c.customer_name,
  c.risk_tolerance,
  h.stock_code,
  mp.stock_name,
  mp.sector,
  h.quantity,
  h.avg_purchase_price,
  mp.closing_price AS current_price,
  h.quantity * h.avg_purchase_price AS investment_value,
  h.quantity * mp.closing_price AS current_value,
  ROUND((mp.closing_price - h.avg_purchase_price) / h.avg_purchase_price * 100, 2) AS return_pct
FROM SECURITIES_WORKSHOP.RAW_DATA.CUSTOMERS c
JOIN SECURITIES_WORKSHOP.RAW_DATA.HOLDINGS h ON c.customer_id = h.customer_id
JOIN SECURITIES_WORKSHOP.ANALYTICS.V_LATEST_PRICES mp ON h.stock_code = mp.stock_code
ORDER BY c.customer_id, h.stock_code;

In [ ]:
df = portfolio_data

segment_summary = df.groupby('RISK_TOLERANCE').agg(
    avg_return=('RETURN_PCT', 'mean'),
    total_investment=('INVESTMENT_VALUE', 'sum'),
    total_current=('CURRENT_VALUE', 'sum'),
    stock_count=('STOCK_CODE', 'count')
).reset_index()

segment_summary['overall_return_pct'] = (
    (segment_summary['total_current'] - segment_summary['total_investment']) 
    / segment_summary['total_investment'] * 100
).round(2)

print("=== 顧客セグメント別サマリー ===")
print(segment_summary.to_string(index=False))

In [ ]:
risk_label = {'バランス': 'Balanced', '保守的': 'Conservative', '積極的': 'Aggressive'}
colors_map = {'Balanced': '#3498db', 'Conservative': '#2ecc71', 'Aggressive': '#e74c3c'}

df_plot = df.copy()
df_plot['RISK_LABEL'] = df_plot['RISK_TOLERANCE'].map(risk_label).fillna(df_plot['RISK_TOLERANCE'])
seg_plot = segment_summary.copy()
seg_plot['RISK_LABEL'] = seg_plot['RISK_TOLERANCE'].map(risk_label).fillna(seg_plot['RISK_TOLERANCE'])

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

seg_sorted = seg_plot.sort_values('overall_return_pct', ascending=True)
bar_colors = [colors_map.get(rt, '#95a5a6') for rt in seg_sorted['RISK_LABEL']]
axes[0].barh(seg_sorted['RISK_LABEL'], seg_sorted['overall_return_pct'], color=bar_colors)
axes[0].set_xlabel('Return (%)')
axes[0].set_title('Segment Performance')
axes[0].axvline(x=0, color='black', linestyle='-', linewidth=0.5)
for i, v in enumerate(seg_sorted['overall_return_pct']):
    axes[0].text(v + 0.3, i, f'{v:.1f}%', va='center', fontweight='bold')

for risk_type in df_plot['RISK_LABEL'].unique():
    subset = df_plot[df_plot['RISK_LABEL'] == risk_type]
    axes[1].scatter(subset['INVESTMENT_VALUE'], subset['RETURN_PCT'], 
                   label=risk_type, alpha=0.7, s=100, color=colors_map.get(risk_type, '#95a5a6'))
axes[1].set_xlabel('Investment Value')
axes[1].set_ylabel('Return (%)')
axes[1].set_title('Investment vs Return')
axes[1].legend()
axes[1].axhline(y=0, color='red', linestyle='--', alpha=0.5)

axes[2].pie(seg_plot['total_investment'], labels=seg_plot['RISK_LABEL'],
           autopct='%1.1f%%', colors=[colors_map.get(rt, '#95a5a6') for rt in seg_plot['RISK_LABEL']],
           startangle=90)
axes[2].set_title('Investment Allocation')

plt.tight_layout()
plt.show()

## 3. ポートフォリオ分散度ヒートマップ
顧客ごとの業種別保有比率をヒートマップで可視化します。ハーフィンダール指数で集中度を測定します。

In [ ]:
sector_pivot = df.pivot_table(
    index=['CUSTOMER_ID', 'CUSTOMER_NAME'], 
    columns='SECTOR', 
    values='CURRENT_VALUE', 
    aggfunc='sum', 
    fill_value=0
)

sector_pct = sector_pivot.div(sector_pivot.sum(axis=1), axis=0) * 100

hhi = (sector_pct ** 2).sum(axis=1)
sector_pct['HHI'] = hhi

print("=== 業種別保有比率 (%) ===")
print(sector_pct.round(1).to_string())
print(f"\nHHI（ハーフィンダール指数）: 低いほど分散、高いほど集中")
print(f"  完全分散（5業種均等）: {5 * (20**2):.0f}")
print(f"  完全集中（1業種のみ）: 10000")

In [ ]:
heatmap_data = sector_pct.drop(columns='HHI')
heatmap_data.index = [cid for cid, cname in heatmap_data.index]

fig = go.Figure(data=go.Heatmap(
    z=heatmap_data.values,
    x=heatmap_data.columns.tolist(),
    y=heatmap_data.index.tolist(),
    text=heatmap_data.round(1).values,
    texttemplate='%{text:.1f}',
    colorscale='YlOrRd',
    colorbar=dict(title='保有比率 (%)'),
    hoverongaps=False
))

fig.update_layout(
    title='顧客×業種 ポートフォリオ分散度ヒートマップ',
    xaxis_title='業種',
    yaxis_title='顧客',
    height=600,
    yaxis=dict(autorange='reversed')
)
fig.show()

## 4. リバランスが必要な顧客の抽出
目標配分比率との乖離が±5%以上の顧客・業種を特定します。

In [ ]:
%%sql -r rebalance_data
SELECT 
  ca.customer_id,
  c.customer_name,
  c.risk_tolerance,
  ca.sector,
  ROUND(ca.allocation_ratio, 2) AS actual_pct,
  ROUND(ca.target_ratio, 2) AS target_pct,
  ROUND(ca.deviation, 2) AS deviation,
  ca.rebalance_flag
FROM SECURITIES_WORKSHOP.ANALYTICS.CUSTOMER_ALLOCATION ca
JOIN SECURITIES_WORKSHOP.RAW_DATA.CUSTOMERS c ON ca.customer_id = c.customer_id
WHERE ca.snapshot_date = CURRENT_DATE()
  AND ABS(ca.deviation) >= 5
ORDER BY ABS(ca.deviation) DESC;

In [ ]:
rebalance_df = rebalance_data

if len(rebalance_df) > 0:
    fig = px.bar(rebalance_df, 
                 x='CUSTOMER_ID', 
                 y='DEVIATION', 
                 color='SECTOR',
                 barmode='group',
                 title='リバランスが必要な顧客・業種（乖離±5%以上）',
                 labels={'DEVIATION': '目標比率との乖離 (%)', 'CUSTOMER_ID': '顧客ID'},
                 hover_data=['CUSTOMER_NAME', 'ACTUAL_PCT', 'TARGET_PCT'],
                 color_discrete_sequence=px.colors.qualitative.Set2)
    fig.add_hline(y=5, line_dash="dash", line_color="red", annotation_text="+5%閾値")
    fig.add_hline(y=-5, line_dash="dash", line_color="red", annotation_text="-5%閾値")
    fig.add_hline(y=0, line_color="black", line_width=0.5)
    fig.update_layout(height=500)
    fig.show()
    
    print(f"\nリバランス対象: {rebalance_df['CUSTOMER_ID'].nunique()}名の顧客、{len(rebalance_df)}件の乖離")
else:
    print("リバランスが必要な顧客はいません（全顧客が目標±5%以内）")

## 5. 業種別パフォーマンス分析
業種ごとの平均リターン、投資額、銘柄数を分析します。

In [ ]:
sector_perf = df.groupby('SECTOR').agg(
    avg_return=('RETURN_PCT', 'mean'),
    total_investment=('INVESTMENT_VALUE', 'sum'),
    total_current=('CURRENT_VALUE', 'sum'),
    stock_count=('STOCK_CODE', 'nunique'),
    holder_count=('CUSTOMER_ID', 'nunique')
).reset_index()

sector_perf['overall_return'] = (
    (sector_perf['total_current'] - sector_perf['total_investment']) 
    / sector_perf['total_investment'] * 100
).round(2)

print("=== 業種別パフォーマンス ===")
print(sector_perf.sort_values('overall_return', ascending=False).to_string(index=False))

In [ ]:
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('業種別 平均リターン', '業種別 投資金額', '業種別 保有顧客数', '投資金額 vs リターン'),
    specs=[[{'type': 'bar'}, {'type': 'pie'}], [{'type': 'bar'}, {'type': 'scatter'}]]
)

sorted_perf = sector_perf.sort_values('overall_return', ascending=True)
colors_return = ['#e74c3c' if x < 0 else '#2ecc71' for x in sorted_perf['overall_return']]
fig.add_trace(
    go.Bar(x=sorted_perf['overall_return'], y=sorted_perf['SECTOR'], orientation='h',
           marker_color=colors_return, name='リターン',
           text=[f"{v:.1f}%" for v in sorted_perf['overall_return']], textposition='outside'),
    row=1, col=1
)

fig.add_trace(
    go.Pie(labels=sector_perf['SECTOR'], values=sector_perf['total_investment'],
           textinfo='label+percent', name='投資額'),
    row=1, col=2
)

fig.add_trace(
    go.Bar(x=sector_perf['SECTOR'], y=sector_perf['holder_count'],
           marker_color='#3498db', name='顧客数',
           text=sector_perf['holder_count'], textposition='outside'),
    row=2, col=1
)

fig.add_trace(
    go.Scatter(x=sector_perf['total_investment'], y=sector_perf['overall_return'],
              mode='markers+text', text=sector_perf['SECTOR'], textposition='top center',
              marker=dict(size=sector_perf['stock_count']*15, color=sector_perf['overall_return'],
                         colorscale='RdYlGn', showscale=True),
              name='業種'),
    row=2, col=2
)

fig.update_layout(height=800, title_text='業種別パフォーマンス ダッシュボード', showlegend=False)
fig.show()
print("\n業種別パフォーマンスダッシュボードを表示しました")

## 6. 統合ダッシュボード
全分析結果を1つのダッシュボードにまとめます。

In [ ]:
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        'セグメント別運用成績',
        '業種別投資配分', 
        '顧客別分散度（HHI）',
        '投資額vsリターン'
    ),
    specs=[[{'type': 'bar'}, {'type': 'pie'}],
           [{'type': 'bar'}, {'type': 'scatter'}]]
)

fig.add_trace(
    go.Bar(x=segment_summary['RISK_TOLERANCE'], y=segment_summary['overall_return_pct'],
           marker_color=['#2ecc71', '#e74c3c', '#3498db'],
           text=[f"{v:.1f}%" for v in segment_summary['overall_return_pct']],
           textposition='outside', name='運用成績'),
    row=1, col=1
)

fig.add_trace(
    go.Pie(labels=sector_perf['SECTOR'], values=sector_perf['total_investment'],
           textinfo='label+percent', name='業種配分'),
    row=1, col=2
)

hhi_data = sector_pct['HHI'].reset_index()
hhi_data.columns = ['CUSTOMER_ID', 'CUSTOMER_NAME', 'HHI']
hhi_data['label'] = hhi_data['CUSTOMER_ID']
fig.add_trace(
    go.Bar(x=hhi_data['label'], y=hhi_data['HHI'],
           marker_color=['#e74c3c' if h > 3000 else '#f39c12' if h > 2500 else '#2ecc71' for h in hhi_data['HHI']],
           text=[f"{v:.0f}" for v in hhi_data['HHI']], textposition='outside', name='HHI'),
    row=2, col=1
)
fig.add_shape(
    type='line', x0=0, x1=1, y0=2500, y1=2500,
    xref='x2 domain', yref='y2',
    line=dict(dash='dash', color='orange')
)

for risk_type in df['RISK_TOLERANCE'].unique():
    subset = df[df['RISK_TOLERANCE'] == risk_type]
    fig.add_trace(
        go.Scatter(x=subset['INVESTMENT_VALUE'], y=subset['RETURN_PCT'],
                  mode='markers', name=risk_type,
                  marker=dict(size=10, opacity=0.7)),
        row=2, col=2
    )

fig.update_layout(
    height=900, 
    title_text='ポートフォリオ分析 統合ダッシュボード',
    title_font_size=20,
    showlegend=True
)
fig.show()

## まとめ

このNotebookで実施した分析：

| 分析 | 手法 | 可視化 |
|---|---|---|
| セグメント別運用成績 | SQL + Pandas集計 | Matplotlib (横棒・散布図・円) |
| ポートフォリオ分散度 | HHI算出 | Seaborn ヒートマップ |
| リバランス判定 | 目標比率±5%閾値 | Plotly インタラクティブ棒グラフ |
| 業種別パフォーマンス | 多角的集計 | Plotly サブプロット |
| 統合ダッシュボード | 全指標統合 | Plotly 4象限ダッシュボード |